# Hafta 14 — Gömülü YZ, LLM'ler, Güvenilirlik ve Etik: Uygulama Defteri

Veri: `motor_ariza.csv` (6. hafta). Bölümler: TinyML (int8, C dizisi, ONNX, budama) · öz-dikkat elle · kalibrasyon · kayma · FGSM · adillik.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F, time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss, recall_score
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
torch.manual_seed(0); np.random.seed(0)
df = pd.read_csv("motor_ariza.csv"); OZ = ["calisma_saati", "titresim_rms_mms", "titresim_kurtosis", "sicaklik_C", "akim_dengesizlik_pct"]
X = df[OZ].values.astype(np.float32); y = df.ariza.values; tip = df.motor_tipi.values
itr, ite = train_test_split(np.arange(len(df)), test_size=0.3, random_state=0, stratify=y)
Xtr, Xte, ytr, yte, tip_te = X[itr], X[ite], y[itr], y[ite], tip[ite]
sc = StandardScaler().fit(Xtr); Xtr_s, Xte_s = sc.transform(Xtr).astype(np.float32), sc.transform(Xte).astype(np.float32)
print(Xtr.shape, Xte.shape, "arıza oranı:", round(y.mean(), 3))

## 1. Küçük MLP eğit: 5 → 8 → 1

In [ ]:
def mlp_egit(H=8, epoch=400, lr=1e-2, seed=0):
    torch.manual_seed(seed); m = nn.Sequential(nn.Linear(5, H), nn.ReLU(), nn.Linear(H, 1)); opt = torch.optim.Adam(m.parameters(), lr)
    Xt, yt = torch.tensor(Xtr_s), torch.tensor(ytr, dtype=torch.float32)
    for ep in range(epoch): opt.zero_grad(); F.binary_cross_entropy_with_logits(m(Xt).squeeze(1), yt).backward(); opt.step()
    return m
def auc(m, Xs=None):
    with torch.no_grad(): return roc_auc_score(yte, m(torch.tensor(Xte_s if Xs is None else Xs)).squeeze(1).numpy())
model = mlp_egit(8); npar = sum(p.numel() for p in model.parameters())
print("parametre:", npar, " float32 bayt:", npar*4, " int8 bayt:", npar, " test AUC:", round(auc(model), 4))
print("MAC sayısı:", 5*8 + 8*1)

## 2. int8 kuantizasyon (Örnek 14.1) ve numpy ile tamsayı çıkarım

In [ ]:
w = np.array([-0.62, -0.10, 0.05, 0.31, 0.88]); s = (w.max() - w.min())/255; z = int(round(-w.min()/s))
q = np.clip(np.round(w/s) + z, 0, 255).astype(np.uint8); print("s =", round(s, 6), "z =", z, "q =", q, "hata =", np.abs((q.astype(float) - z)*s - w).round(4))
# modelin ağırlıklarını simetrik int8'e çevir (kanal başına ölçek)
def kuantize(Wf):
    s_ = np.abs(Wf).max(axis=1, keepdims=True)/127; return np.round(Wf/s_).astype(np.int8), s_.astype(np.float32)
W1, b1 = model[0].weight.detach().numpy(), model[0].bias.detach().numpy(); W2, b2 = model[2].weight.detach().numpy(), model[2].bias.detach().numpy()
q1, s1 = kuantize(W1); q2, s2 = kuantize(W2)
def int8_cikarim(Xs):                                  # ağırlık int8, aktivasyon float (ağırlık-only kuantizasyon)
    h = np.maximum(0, Xs @ (q1.astype(np.float32)*s1).T + b1); return h @ (q2.astype(np.float32)*s2).T + b2
pr_f = model(torch.tensor(Xte_s)).detach().numpy().ravel(); pr_q = int8_cikarim(Xte_s).ravel()
print("AUC float32 %.4f | int8 %.4f | maks logit farkı %.4f" % (roc_auc_score(yte, pr_f), roc_auc_score(yte, pr_q), np.abs(pr_f - pr_q).max()))
print("W1 int8:\n", q1)

**Soru:** Girdiyi de int8 yapmak isteseniz (tam tamsayı çıkarım) ölçeği nereden bulursunuz? (İpucu: kalibrasyon verisi ile aktivasyon aralığı.)

## 3. Ağırlıkları C başlık dosyasına dök

In [ ]:
def c_dizi(ad, A, tip="int8_t"):
    if A.ndim == 1: return f"static const {tip} {ad}[{A.shape[0]}] = {{" + ", ".join(f"{v:.6f}f" if tip == "float" else str(int(v)) for v in A) + "};\n"
    return f"static const {tip} {ad}[{A.shape[0]}][{A.shape[1]}] = {{" + ", ".join("{" + ", ".join(str(int(v)) for v in r) + "}" for r in A) + "};\n"
with open("motor_mlp.h", "w") as f:
    f.write("// motor_ariza MLP 5->8->1, int8 agirlik + float olcek. Girdi: StandardScaler ile olceklenmis 5 ozellik\n#include <stdint.h>\n")
    f.write(c_dizi("MU", sc.mean_.astype(np.float32), "float") + c_dizi("SIG", sc.scale_.astype(np.float32), "float"))
    f.write(c_dizi("W1", q1) + c_dizi("S1", s1.ravel(), "float") + c_dizi("B1", b1, "float") + c_dizi("W2", q2) + c_dizi("S2", s2.ravel(), "float") + c_dizi("B2", b2, "float"))
    f.write("""
static float motor_ariza_logit(const float x_raw[5]) {
    float x[5], h[8]; int i, j;
    for (j = 0; j < 5; j++) x[j] = (x_raw[j] - MU[j]) / SIG[j];
    for (i = 0; i < 8; i++) { int32_t acc = 0; for (j = 0; j < 5; j++) acc += (int32_t)W1[i][j] * (int32_t)(x[j] * 127.0f);
                              h[i] = acc * S1[i] / 127.0f + B1[i]; if (h[i] < 0) h[i] = 0; }
    { int32_t acc = 0; for (i = 0; i < 8; i++) acc += (int32_t)W2[0][i] * (int32_t)(h[i] * 127.0f);
      return acc * S2[0] / 127.0f + B2[0]; }
}
""")
print(open("motor_mlp.h").read()[:900]); import os; print("... dosya boyutu:", os.path.getsize("motor_mlp.h"), "bayt (kaynak); çalışma zamanı ~", npar + 8*4 + 10*4, "bayt")

## 4. ONNX'e aktar ve onnxruntime ile gecikme ölç

In [ ]:
try:
    import onnxruntime as ort
    torch.onnx.export(model, torch.zeros(1, 5), "motor_mlp.onnx", input_names=["x"], output_names=["logit"], dynamo=False)
    sess = ort.InferenceSession("motor_mlp.onnx"); x1 = Xte_s[:1]
    t0 = time.perf_counter(); [sess.run(None, {"x": x1}) for _ in range(2000)]; t_onnx = (time.perf_counter() - t0)/2000
    t0 = time.perf_counter(); [model(torch.tensor(x1)) for _ in range(2000)]; t_torch = (time.perf_counter() - t0)/2000
    t0 = time.perf_counter(); [int8_cikarim(x1) for _ in range(2000)]; t_np = (time.perf_counter() - t0)/2000
    print(f"tek örnek gecikme — PyTorch {t_torch*1e6:.0f} µs | onnxruntime {t_onnx*1e6:.0f} µs | numpy int8 {t_np*1e6:.0f} µs   (PC'de; MCU'da 100 MHz ≈ 1–5 µs hesap)")
    print("ONNX dosyası:", os.path.getsize("motor_mlp.onnx"), "bayt")
except Exception as e: print("ONNX adımı atlandı:", e)

## 5. Öz-dikkat elle (Örnek 14.3) ve minik bir dil modeli

In [ ]:
Q = K = np.array([[1, 0], [0, 1], [1, 1.]]); V = np.array([[1, 0], [0, 1], [.5, .5]])
S = Q @ K.T/np.sqrt(2); A = np.exp(S)/np.exp(S).sum(1, keepdims=True); print("dikkat ağırlıkları:\n", A.round(3)); print("çıktı:\n", (A @ V).round(3))
# PyTorch ile aynı şey
At = F.scaled_dot_product_attention(torch.tensor(Q)[None], torch.tensor(K)[None], torch.tensor(V)[None]); print("torch:", At[0].numpy().round(3).tolist())

In [ ]:
# 'sonraki token' fikri en küçük hâliyle: karakter bigram modeli (sayım tablosu = 1 katmanlı 'LLM')
metin = ("transformator sargisi asiri isindi. kesici acildi. rulman titresimi artti. motor akimi dengesiz. "
         "inverter cikis gerilimi dustu. pv uretimi bulutla azaldi. sebeke frekansi sapti. rolenin ayari degisti. ")*3
kar = sorted(set(metin)); ix = {c: i for i, c in enumerate(kar)}; N = np.ones((len(kar), len(kar)))          # Laplace düzeltmesi
for a, b in zip(metin[:-1], metin[1:]): N[ix[a], ix[b]] += 1
P = N/N.sum(1, keepdims=True); rng = np.random.default_rng(3); c = ix["m"]; cikti = "m"
for _ in range(80): c = rng.choice(len(kar), p=P[c]); cikti += kar[c]
print(cikti); print("kayıp (bit/karakter):", round(-np.mean([np.log2(P[ix[a], ix[b]]) for a, b in zip(metin[:-1], metin[1:])]), 3), " rastgele:", round(np.log2(len(kar)), 3))

**Soru:** Bigram modeli neden anlamsız kelimeler üretiyor? Bağlam uzunluğunu 1'den 1000'e çıkarınca ne değişir — ve tablo neden artık kullanılamaz (LLM'nin tablo yerine ağ kullanmasının nedeni)?

## 6. Kalibrasyon: güvenilirlik diyagramı, Brier, ECE

In [ ]:
def ece(y, p, k=8):
    kut = np.quantile(p, np.linspace(0, 1, k + 1)); kut[-1] += 1e-9; e = 0
    for a, b in zip(kut[:-1], kut[1:]):
        mk = (p >= a) & (p < b)
        if mk.sum(): e += mk.mean()*abs(y[mk].mean() - p[mk].mean())
    return e
rf = RandomForestClassifier(200, random_state=0).fit(Xtr, ytr); lr = LogisticRegression(max_iter=1000).fit(Xtr_s, ytr)
with torch.no_grad(): p_mlp = torch.sigmoid(model(torch.tensor(Xte_s))).numpy().ravel()
modeller = {"orman": rf.predict_proba(Xte)[:, 1], "lojistik": lr.predict_proba(Xte_s)[:, 1], "MLP": p_mlp}
plt.figure(figsize=(5, 4.2)); plt.plot([0, 1], [0, 1], "--", color="gray")
for ad, p in modeller.items():
    fr, mp = calibration_curve(yte, p, n_bins=8, strategy="quantile"); plt.plot(mp, fr, "o-", label=ad)
    print(f"{ad:9s} AUC {roc_auc_score(yte, p):.3f}  Brier {brier_score_loss(yte, p):.4f}  ECE {ece(yte, p):.4f}")
plt.xlabel("tahmin"); plt.ylabel("gözlenen oran"); plt.legend(); plt.title("Güvenilirlik diyagramı"); plt.show()
# Örnek 14.4
n = np.array([400, 300, 200, 100]); tah = np.array([.05, .25, .55, .85]); sik = np.array([.03, .20, .65, .70]); print("Örnek 14.4 ECE =", round(np.sum(n*np.abs(sik - tah))/n.sum(), 3))

**Soru:** Ormanı `CalibratedClassifierCV(method='isotonic', cv=5)` ile sarın; Brier ve ECE değişir mi, AUC değişir mi? Neden AUC (sıralama) kalibrasyondan bağımsızdır?

## 7. Dağılım kayması deneyi

In [ ]:
from sklearn.metrics import f1_score
kaymalar = np.arange(0, 21, 2); sonuc = {"orman": [], "lojistik": [], "MLP": []}
for k in kaymalar:
    Xk = Xte.copy(); Xk[:, 3] += k; Xk_s = sc.transform(Xk).astype(np.float32)
    with torch.no_grad(): p_m = torch.sigmoid(model(torch.tensor(Xk_s))).numpy().ravel()
    for ad, pk in (("orman", rf.predict_proba(Xk)[:, 1]), ("lojistik", lr.predict_proba(Xk_s)[:, 1]), ("MLP", p_m)):
        sonuc[ad].append((roc_auc_score(yte, pk), f1_score(yte, pk > 0.3), (pk > 0.3).mean()))
fig, axs = plt.subplots(1, 3, figsize=(12, 3.2))
for j, t in enumerate(("AUC", "F1 @ 0.3", "alarm oranı @ 0.3")):
    for ad, v in sonuc.items(): axs[j].plot(kaymalar, [r[j] for r in v], "o-", label=ad)
    axs[j].set_title(t); axs[j].set_xlabel("sıcaklık kayması (°C)"); axs[j].grid(alpha=.3)
axs[0].legend(); plt.tight_layout(); plt.show()
print("orman: kayma 0 → 20 °C: AUC %.3f → %.3f | F1 %.2f → %.2f | alarm %%%.0f → %%%.0f" % (sonuc["orman"][0][0], sonuc["orman"][-1][0], sonuc["orman"][0][1], sonuc["orman"][-1][1], 100*sonuc["orman"][0][2], 100*sonuc["orman"][-1][2]))
# basit izleme: özellik z-skoru
z_uretim = (Xte[:, 3] + 12 - sc.mean_[3])/sc.scale_[3]; print("üretimde sıcaklık z-skoru ortalaması: %.2f (eğitimde 0) → alarm eşiği |z̄| > 0.5" % z_uretim.mean())

**Soru:** AUC neden neredeyse değişmiyor ama F1 ve alarm oranı bozuluyor? (AUC sıralamayı ölçer; kayma tüm olasılıkları birlikte yukarı iter.) Üretimde yalnızca AUC izleseydiniz kaymayı fark eder miydiniz? Hangi model daha dayanıklı — ağaçlar eğitim aralığının dışında sabit kalır, doğrusal model dışarı doğru uzatır.

## 8. Düşmanca örnek: FGSM (rakam MLP'si)

In [ ]:
from sklearn.datasets import load_digits
D = load_digits(); Xd = (D.data/16).astype(np.float32); yd = D.target; Xa, Xb, ya, yb = train_test_split(Xd, yd, test_size=0.3, random_state=0)
torch.manual_seed(0); md_ = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10)); opt = torch.optim.Adam(md_.parameters(), 3e-3); Xa_t, ya_t = torch.tensor(Xa), torch.tensor(ya)
for ep in range(300): opt.zero_grad(); F.cross_entropy(md_(Xa_t), ya_t).backward(); opt.step()
Xb_t = torch.tensor(Xb, requires_grad=True); yb_t = torch.tensor(yb); F.cross_entropy(md_(Xb_t), yb_t).backward(); g = Xb_t.grad.sign()
for e in (0, .02, .05, .1, .2):
    with torch.no_grad(): print(f"ε = {e:.2f}  doğruluk {(md_((Xb_t + e*g).clamp(0, 1)).argmax(1) == yb_t).float().mean():.3f}")
i = 3; fig, ax = plt.subplots(1, 3, figsize=(7, 2.5))
for a, e in zip(ax, (0, .1, .2)):
    xi = (Xb_t[i] + e*g[i]).clamp(0, 1).detach(); a.imshow(xi.reshape(8, 8), cmap="gray"); a.set_title(f"ε={e} → {md_(xi[None]).argmax().item()} (gerçek {yb[i]})", fontsize=9); a.axis("off")
plt.show()

**Soru:** Rastgele gürültü (aynı ε, işaretleri rastgele) aynı etkiyi yapar mı? Deneyin ve farkı açıklayın.

## 9. Adillik: alt grup raporu

In [ ]:
p_rf = rf.predict_proba(Xte)[:, 1]; esik = 0.3
print(f"{'grup':10s} {'n':>4s} {'arıza%':>7s} {'TPR':>6s} {'FPR':>6s} {'AUC':>6s}")
for t in ("asenkron", "senkron", "dc"):
    mk = tip_te == t; yh = p_rf[mk] > esik
    print(f"{t:10s} {mk.sum():4d} {100*yte[mk].mean():7.1f} {recall_score(yte[mk], yh):6.2f} {((yh == 1) & (yte[mk] == 0)).sum()/(yte[mk] == 0).sum():6.2f} {roc_auc_score(yte[mk], p_rf[mk]):6.3f}")

**Soru:** Gruplar arasındaki fark istatistiksel olarak anlamlı mı (n küçük!)? Grup başına eşik seçmek adil midir — hangi 'adillik' tanımına göre (eşit TPR mi, eşit FPR mi, eşit kalibrasyon mu)?

## 10. Alıştırmalar

**Alıştırma 1 (budama).** MLP'nin (H = 32 ile yeniden eğitin) mutlak değerce en küçük %50, %70, %90 ağırlığını sıfırlayın; AUC'yi ölçün. Sıfırlanmış ağırlıkları seyrek (CSR) saklasanız bellek ne olur?

**Alıştırma 2 (belirsizlik).** 10 farklı tohumla MLP eğitin; test örnekleri için tahmin ortalaması ve standart sapmasını hesaplayın. En belirsiz 10 örneğe bakın — bunlar sınırda mı? Alternatif: dropout'lu MLP ile MC dropout (50 geçiş).

**Alıştırma 3 (konformal).** Orman olasılığı için doğrulama kümesinde uygunsuzluk skoru 1 − p_y hesaplayın; %90 kantili q ile test örneklerine tahmin kümesi {sınıf : 1 − p ≤ q} atayın. Kapsama oranı ≈ 0.90 mı? Kaç örnek 'her iki sınıf' (kararsız) aldı?

**Alıştırma 4 (damıtma).** Ormanın olasılıklarını 'yumuşak etiket' olarak kullanıp H = 4 MLP'yi BCE ile eğitin; sert etiketle eğitilmiş H = 4 MLP'ye karşı AUC.

**Alıştırma 5 (model kartı).** Bu defterdeki orman modeli için 10 satırlık bir model kartı yazın: amaç, veri, metrikler (alt gruplarla), kalibrasyon, kayma sınırı, kullanım dışı durumlar.